# Train / Validation / Test Split

Hold out data for **unbiased evaluation** — typically 70/15/15 or 80/10/10.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11
torch.manual_seed(42)
np.random.seed(42)

class DummyDataGenerator:
    """Synthetic classification data for CPU-only tutorials."""
    def __init__(self, n_samples=256, n_features=8, n_classes=3, seed=42):
        g = torch.Generator().manual_seed(seed)
        self.X = torch.randn(n_samples, n_features, generator=g)
        self.y = torch.randint(0, n_classes, (n_samples,), generator=g)

    def tensors(self):
        return self.X, self.y

class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


## 1. Split indices

In [ ]:
gen = DummyDataGenerator(n_samples=500, n_features=8, n_classes=3)
X, y = gen.tensors()
n = len(y)
n_train = int(0.7 * n)
n_val = int(0.15 * n)
perm = torch.randperm(n)
train_idx = perm[:n_train]
val_idx = perm[n_train:n_train + n_val]
test_idx = perm[n_train + n_val:]
print(f"train={len(train_idx)}, val={len(val_idx)}, test={len(test_idx)}")

## 2. Pie chart of splits

In [ ]:
sizes = [len(train_idx), len(val_idx), len(test_idx)]
labels = ['Train 70%', 'Val 15%', 'Test 15%']
colors = ['#3498db', '#f39c12', '#e74c3c']
fig, ax = plt.subplots(figsize=(7, 6))
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax.set_title('Dataset split proportions')
plt.tight_layout(); plt.show()

## 3. Class balance per split

In [ ]:
def class_hist(idx, title):
    counts = torch.bincount(y[idx], minlength=3).numpy()
    return counts

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, idx, name in zip(axes, [train_idx, val_idx, test_idx], ['Train', 'Val', 'Test']):
    c = class_hist(idx, name)
    ax.bar(range(3), c, color='teal', edgecolor='black')
    ax.set_title(f'{name} class counts'); ax.set_xlabel('class')
plt.tight_layout(); plt.show()

## 4. Create DataLoaders per split

In [ ]:
train_ds = TabularDataset(X[train_idx], y[train_idx])
val_ds = TabularDataset(X[val_idx], y[val_idx])
test_ds = TabularDataset(X[test_idx], y[test_idx])
loaders = {
    'train': DataLoader(train_ds, batch_size=32, shuffle=True),
    'val': DataLoader(val_ds, batch_size=32),
    'test': DataLoader(test_ds, batch_size=32),
}
for k, ld in loaders.items():
    print(f"{k}: {len(ld)} batches")

## 5. Sample counts bar chart

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['Train', 'Val', 'Test'], sizes, color=colors, edgecolor='black')
ax.set_ylabel('# samples'); ax.set_title('Absolute split sizes')
plt.tight_layout(); plt.show()